# TA-06 — Prediksi Persetujuan Pengajuan Pinjaman (Loan Approval Prediction) Menggunakan Random Forest

**Mata Kuliah:** Pembelajaran Mesin · **Topik:** Eksplorasi Mendalam Algoritma Random Forest

Notebook ini berisi:
1. Persiapan data & preprocessing
2. Skenario A: pengaruh jumlah pohon (`n_estimators`)
3. Skenario B: pengaruh kedalaman pohon (`max_depth`)
4. Analisis Feature Importance
5. Evaluasi akhir & kesimpulan

> Jalankan sel dari atas ke bawah (**Run All**). Semua grafik juga disimpan ke folder `reports/figures/` untuk dipakai di laporan.

## 1. Import Library

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42          # agar hasil selalu sama setiap dijalankan
FIG_DIR = "../reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)

## 2. Persiapan Data & Preprocessing

### 2.1 Load data
Dataset: **Loan Approval Prediction Dataset** (4.269 baris, 13 kolom). Target: `loan_status` (*Approved* / *Rejected*).

In [ ]:
df = pd.read_csv("../data/loan_approval_dataset.csv")
print("Ukuran data:", df.shape)
print("Nama kolom asli:", list(df.columns))
df.head()

### 2.2 Pembersihan data
Pemeriksaan awal menemukan beberapa masalah:
- **Nama kolom diawali spasi** (misalnya `' loan_status'`), sehingga `df['loan_status']` akan error.
- **Nilai teks juga diawali spasi** (misalnya `' Approved'`).
- `loan_id` hanya nomor urut dan tidak punya makna prediktif.
- Ada nilai **`residential_assets_value` negatif**. Nilai aset tidak mungkin negatif, jadi dianggap salah input dan diganti 0.

In [ ]:
# 1) Hapus spasi di nama kolom
df.columns = df.columns.str.strip()

# 2) Hapus spasi di nilai teks
for col in ["education", "self_employed", "loan_status"]:
    df[col] = df[col].astype(str).str.strip()

# 3) Buang kolom ID
df = df.drop(columns=["loan_id"])

# 4) Cek missing value & duplikat
print("Missing value per kolom:\n", df.isna().sum(), "\n")
print("Jumlah baris duplikat:", df.duplicated().sum())

# 5) Nilai aset negatif -> 0
print("Aset rumah negatif:", (df["residential_assets_value"] < 0).sum(), "baris")
df["residential_assets_value"] = df["residential_assets_value"].clip(lower=0)

df.info()

In [ ]:
df.describe().T

### 2.3 Analisis awal: cek ketidakseimbangan kelas (imbalance)

In [ ]:
dist = df["loan_status"].value_counts()
print(dist)
print("\nPersentase:\n", (dist / len(df) * 100).round(2))
print(f"\nRasio mayoritas : minoritas = {dist.max() / dist.min():.2f} : 1")

plt.figure(figsize=(6, 4))
ax = sns.countplot(x=df["loan_status"], hue=df["loan_status"], palette=["#2e7d32", "#c62828"], legend=False)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.title("Distribusi Status Pinjaman")
plt.xlabel("Status"); plt.ylabel("Jumlah")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/distribusi_kelas.png", dpi=120); plt.show()

**Interpretasi:** *Approved* sekitar 62% dan *Rejected* sekitar 38% (rasio ±1,65 : 1). Ini **ketidakseimbangan ringan**, sehingga tidak perlu resampling. Namun:
- saat split dipakai `stratify=y` agar proporsi kelas di data latih dan uji tetap sama;
- selain akurasi, kita juga melihat **F1-macro** dan **recall per kelas**.

### 2.4 Encoding data kategori
Kolom kategori hanya punya 2 nilai, sehingga cukup **Label Encoding** (0/1). Random Forest tidak membutuhkan One-Hot Encoding maupun normalisasi/scaling karena model berbasis pohon hanya membandingkan nilai dengan ambang batas.

In [ ]:
df_enc = df.copy()
df_enc["education"] = df_enc["education"].map({"Graduate": 1, "Not Graduate": 0})
df_enc["self_employed"] = df_enc["self_employed"].map({"Yes": 1, "No": 0})
df_enc["loan_status"] = df_enc["loan_status"].map({"Approved": 1, "Rejected": 0})
df_enc.head()

### 2.5 Split data 80% latih : 20% uji

In [ ]:
X = df_enc.drop(columns=["loan_status"])
y = df_enc["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Data latih:", X_train.shape, "| Data uji:", X_test.shape)
print("Proporsi Approved di latih: %.3f | di uji: %.3f" % (y_train.mean(), y_test.mean()))

## 3. Skenario A: Pengaruh Jumlah Pohon (`n_estimators`)
Variasi: **[10, 50, 100, 200, 500]**. Selain akurasi testing (sesuai instruksi), dihitung juga **akurasi cross-validation 5-fold pada data latih** sebagai ukuran stabilitas yang tidak bergantung pada satu kali split.

In [ ]:
n_list = [10, 50, 100, 200, 500]
hasil_a = []

for n in n_list:
    rf = RandomForestClassifier(n_estimators=n, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_train, y_train)
    acc_train = accuracy_score(y_train, rf.predict(X_train))
    acc_test = accuracy_score(y_test, rf.predict(X_test))
    cv = cross_val_score(RandomForestClassifier(n_estimators=n, random_state=RANDOM_STATE, n_jobs=-1),
                         X_train, y_train, cv=5, scoring="accuracy")
    hasil_a.append({"n_estimators": n, "acc_train": acc_train, "acc_test": acc_test,
                    "cv_mean": cv.mean(), "cv_std": cv.std()})

df_a = pd.DataFrame(hasil_a)
df_a.round(4)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(df_a["n_estimators"], df_a["acc_test"], marker="o", linewidth=2, label="Akurasi Testing")
plt.plot(df_a["n_estimators"], df_a["cv_mean"], marker="s", linestyle="--", label="Akurasi CV 5-fold (train)")
for x_, y_ in zip(df_a["n_estimators"], df_a["acc_test"]):
    plt.annotate(f"{y_:.4f}", (x_, y_), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
plt.xscale("log"); plt.xticks(n_list, [str(n) for n in n_list])
plt.xlabel("Jumlah Pohon (n_estimators)"); plt.ylabel("Akurasi")
plt.title("Skenario A: Pengaruh Jumlah Pohon terhadap Akurasi")
plt.legend(); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/skenario_a.png", dpi=120); plt.show()

# Pilih n_estimators terbaik: akurasi test tertinggi; jika seri, pilih pohon paling sedikit
best_n = int(df_a[df_a["acc_test"] == df_a["acc_test"].max()]["n_estimators"].min())
print("n_estimators terbaik:", best_n)

**Analisis Skenario A** *(angka dari hasil run penulis; bisa sedikit berbeda jika versi library berbeda)*

| n_estimators | Akurasi Test | CV 5-fold |
|---|---|---|
| 10 | 0,9813 | 0,9719 ± 0,0034 |
| 50 | 0,9801 | 0,9775 ± 0,0047 |
| 100 | 0,9801 | 0,9798 ± 0,0049 |
| **200** | **0,9824** | 0,9789 ± 0,0055 |
| 500 | 0,9813 | 0,9816 ± 0,0048 |

- **Apakah menambah pohon terus-menerus meningkatkan akurasi secara signifikan? Tidak.** Akurasi testing hanya bergerak di rentang 0,980–0,982. Selisih 0,0012 setara dengan **1 sampel** dari 854 data uji, jadi perbedaannya tidak bermakna.
- Akurasi CV naik cukup jelas dari 10 ke 100 pohon (0,972 → 0,980), lalu **mendatar (plateau)**. Pola ini sesuai teori: menambah pohon menurunkan variansi, tetapi manfaatnya semakin kecil (*diminishing returns*), sementara waktu training dan ukuran model terus bertambah.
- Akurasi training 10 pohon sudah 0,998 dan ≥50 pohon 1,000. Ini wajar karena pohon tanpa batas kedalaman menghafal data latih. Indikator generalisasi yang benar adalah akurasi **testing/CV**.
- Dipilih **n_estimators = 200** (akurasi testing tertinggi). Pilihan ini cukup stabil dan efisien dibanding 500 pohon.

## 4. Skenario B: Pengaruh Kedalaman Pohon (`max_depth`)
Memakai `n_estimators` terbaik dari Skenario A, dengan variasi **[None, 3, 5, 10, 20]**. Bandingkan akurasi training dan testing untuk mendeteksi overfitting.

In [ ]:
depth_list = [3, 5, 10, 20, None]   # diurutkan dari dangkal ke tak terbatas
hasil_b = []

for d in depth_list:
    rf = RandomForestClassifier(n_estimators=best_n, max_depth=d, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_train, y_train)
    acc_train = accuracy_score(y_train, rf.predict(X_train))
    acc_test = accuracy_score(y_test, rf.predict(X_test))
    hasil_b.append({"max_depth": "None" if d is None else str(d),
                    "acc_train": acc_train, "acc_test": acc_test, "gap": acc_train - acc_test})

df_b = pd.DataFrame(hasil_b)
df_b.round(4)

In [ ]:
x_pos = np.arange(len(df_b)); lebar = 0.38
plt.figure(figsize=(8, 5))
plt.bar(x_pos - lebar/2, df_b["acc_train"], lebar, label="Akurasi Training", color="#1565c0")
plt.bar(x_pos + lebar/2, df_b["acc_test"], lebar, label="Akurasi Testing", color="#ef6c00")
for i, row in df_b.iterrows():
    plt.text(i - lebar/2, row["acc_train"] + 0.003, f"{row['acc_train']:.3f}", ha="center", fontsize=8)
    plt.text(i + lebar/2, row["acc_test"] + 0.003, f"{row['acc_test']:.3f}", ha="center", fontsize=8)
plt.xticks(x_pos, df_b["max_depth"])
plt.ylim(df_b[["acc_train", "acc_test"]].min().min() - 0.05, 1.02)
plt.xlabel("Kedalaman Pohon (max_depth)"); plt.ylabel("Akurasi")
plt.title(f"Skenario B: Training vs Testing (n_estimators={best_n})")
plt.legend(loc="lower right"); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/skenario_b.png", dpi=120); plt.show()

# Pilih max_depth terbaik: akurasi test tertinggi; jika seri, pilih yang lebih dangkal
best_label = df_b[df_b["acc_test"] == df_b["acc_test"].max()]["max_depth"].iloc[0]
best_depth = None if best_label == "None" else int(best_label)
print("max_depth terbaik:", best_label)

**Analisis Skenario B**

| max_depth | Train | Test | Selisih |
|---|---|---|---|
| 3 | 0,9652 | 0,9719 | −0,0067 |
| 5 | 0,9763 | 0,9742 | +0,0020 |
| 10 | 0,9997 | 0,9801 | +0,0196 |
| **20** | 1,0000 | **0,9824** | +0,0176 |
| None | 1,0000 | 0,9824 | +0,0176 |

- **Depth 3–5 (underfitting ringan):** akurasi training dan testing hampir sama, tetapi keduanya lebih rendah. Model terlalu sederhana untuk menangkap seluruh pola.
- **Mulai depth 10, gejala overfitting muncul:** akurasi training melonjak hingga ±1,000, sedangkan testing hanya 0,980. Celah train–test melebar dari 0,002 menjadi ±0,02.
- Walaupun begitu, **akurasi testing tidak turun**, bahkan sedikit naik (0,974 → 0,982). Inilah keunggulan Random Forest: setiap pohon boleh overfit, tetapi *bagging* dan pemilihan fitur acak membuat kesalahan antarpohon saling meniadakan saat voting. Overfitting yang terjadi bersifat **"jinak"** dan tidak merusak generalisasi.
- Depth 20 dan None memberi hasil identik karena pohon pada dataset ini secara alami tidak tumbuh lebih dalam dari ±20 level.
- Dipilih **max_depth = 20**: akurasi testing tertinggi, dengan batas kedalaman eksplisit yang lebih aman daripada *None* jika data bertambah.
- *Catatan:* bila prioritasnya model paling sederhana dengan celah train–test terkecil, **max_depth = 5** adalah alternatif yang baik (akurasi testing 0,974, celah hanya 0,002).

## 5. Model Terbaik & Analisis Feature Importance

In [ ]:
best_model = RandomForestClassifier(n_estimators=best_n, max_depth=best_depth,
                                    random_state=RANDOM_STATE, n_jobs=-1)
best_model.fit(X_train, y_train)

label_id = {
    "no_of_dependents": "Jumlah Tanggungan", "education": "Pendidikan", "self_employed": "Wiraswasta",
    "income_annum": "Pendapatan Tahunan", "loan_amount": "Jumlah Pinjaman", "loan_term": "Tenor Pinjaman",
    "cibil_score": "Skor Kredit (CIBIL)", "residential_assets_value": "Aset Rumah Tinggal",
    "commercial_assets_value": "Aset Komersial", "luxury_assets_value": "Aset Mewah",
    "bank_asset_value": "Aset di Bank",
}
fi = pd.DataFrame({"feature": X.columns, "importance": best_model.feature_importances_})
fi["label"] = fi["feature"].map(label_id)
fi = fi.sort_values("importance", ascending=False).reset_index(drop=True)
fi

In [ ]:
top10 = fi.head(10)
plt.figure(figsize=(9, 6))
sns.barplot(data=top10, x="importance", y="label", hue="label", palette="viridis", legend=False)
for i, v in enumerate(top10["importance"]):
    plt.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
plt.xlabel("Tingkat Kepentingan (Mean Decrease in Impurity)"); plt.ylabel("Fitur")
plt.title("10 Fitur Paling Berpengaruh - Random Forest")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/feature_importance.png", dpi=120); plt.show()

**Interpretasi Feature Importance**

1. **Skor Kredit / CIBIL (±0,80) sangat dominan.** Skor kredit merangkum riwayat pembayaran pemohon. Dalam praktik perbankan, inilah filter pertama: pemohon dengan riwayat buruk hampir pasti ditolak berapa pun pendapatannya. **Sesuai intuisi domain.**
2. **Tenor Pinjaman (±0,065)** dan **Jumlah Pinjaman (±0,031)** berada di urutan berikutnya. Keduanya menentukan besar cicilan dan risiko gagal bayar, sehingga logis menjadi faktor kedua.
3. **Pendapatan dan nilai aset (masing-masing ±0,015–0,02)** berperan kecil. Aset berfungsi sebagai jaminan, tetapi tidak cukup untuk mengubah keputusan jika skor kredit sudah jelas tinggi atau rendah.
4. **Pendidikan, Wiraswasta, dan Jumlah Tanggungan (<0,01)** hampir tidak berpengaruh. Ini juga hal positif, karena keputusan kredit idealnya tidak bergantung pada atribut demografis.

*Catatan metodologis:* *feature importance* berbasis impurity (MDI) cenderung memberi nilai lebih tinggi pada fitur numerik dengan banyak nilai unik. Namun dominasi CIBIL di sini sangat besar (lebih dari 10× fitur kedua), sehingga kesimpulannya tetap kuat.

**Wawasan bisnis (Business Insight):**
- Proses penyaringan awal dapat dipercepat dengan memeriksa skor kredit terlebih dahulu.
- Untuk calon nasabah yang ditolak, saran yang paling berdampak adalah **memperbaiki skor kredit** dan menyesuaikan **jumlah/tenor pinjaman**, bukan sekadar menambah aset.
- Karena fitur demografis tidak berpengaruh, model ini relatif adil dari sisi pendidikan dan status pekerjaan.

## 6. Evaluasi Akhir Model Terbaik

In [ ]:
y_pred = best_model.predict(X_test)
print(f"Konfigurasi : n_estimators={best_n}, max_depth={best_label}")
print(f"Akurasi     : {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-macro    : {f1_score(y_test, y_pred, average='macro'):.4f}\n")
print(classification_report(y_test, y_pred, target_names=["Rejected", "Approved"], digits=4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Rejected", "Approved"], yticklabels=["Rejected", "Approved"])
plt.xlabel("Prediksi"); plt.ylabel("Aktual"); plt.title("Confusion Matrix - Model Terbaik")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/confusion_matrix.png", dpi=120); plt.show()

**Analisis evaluasi** (854 data uji)
- **Akurasi 98,24%** dan **F1-macro 0,9813**. Performa kedua kelas seimbang: F1 *Rejected* 0,977 dan F1 *Approved* 0,986, jadi model tidak bias ke kelas mayoritas.
- **False Positive = 9** (seharusnya ditolak, diprediksi disetujui). Bagi bank, ini kesalahan **paling mahal** karena berisiko kredit macet.
- **False Negative = 6** (seharusnya disetujui, diprediksi ditolak). Dampaknya kehilangan potensi nasabah, relatif lebih ringan.
- Jika bank ingin lebih konservatif, ambang probabilitas persetujuan dapat dinaikkan (misalnya disetujui hanya jika `P(Approved) ≥ 0,7`) untuk menekan False Positive.

## 7. Kesimpulan

1. Random Forest bekerja sangat baik pada dataset ini: **akurasi 98,24%** dan **F1-macro 0,98** pada data uji.
2. **Konfigurasi terbaik dan stabil: `n_estimators = 200`, `max_depth = 20`.**
3. **Skenario A:** menambah pohon di atas ±100 tidak meningkatkan akurasi secara signifikan (plateau ±0,98), tetapi menambah waktu komputasi dan ukuran model.
4. **Skenario B:** kedalaman 3–5 sedikit *underfit*; mulai depth 10 terlihat celah train–test (overfitting), namun akurasi testing tetap tinggi berkat mekanisme *bagging* dan *voting*.
5. **Feature importance:** skor kredit (CIBIL) adalah penentu utama (±80%), disusul tenor dan jumlah pinjaman. Hasil ini sesuai dengan logika penilaian kredit di dunia perbankan.
6. **Keterbatasan:** pemilihan hyperparameter memakai akurasi testing (sesuai instruksi tugas), sehingga data uji ikut memengaruhi pemilihan model. Hasil CV 5-fold pada data latih (±0,98) yang konsisten menunjukkan model tetap stabil.

## 8. Simpan Model (opsional — `train.py` melakukan hal yang sama untuk aplikasi web)

In [ ]:
import joblib
os.makedirs("../model", exist_ok=True)
joblib.dump(best_model, "../model/model_notebook.joblib", compress=3)
print("Model tersimpan di ../model/model_notebook.joblib")